# MedViTV2 — Training Pipeline

The official [`Tutorials/Evaluation.ipynb`](https://github.com/Omid-Nejati/MedViTV2/blob/main/Tutorials/Evaluation.ipynb)
trains with a single line — `!python main.py ...` — which hides the whole pipeline in a subprocess.
This notebook lays the pipeline out as cells instead.

The **reusable pieces** (model selection, the two training routines, the metric helpers and the
all-metrics evaluation) live in [`_handlers/evaluation.py`](../../../_handlers/evaluation.py); the
notebook keeps only the configuration and the linear flow, so the pipeline reads top-to-bottom and
the logic can be reused elsewhere. The layout mirrors
[`Instructions.ipynb`](../classical/Instructions.ipynb).

Use a GPU (a Colab **T4** is enough) — MedViTV2 relies on `natten`, which needs CUDA.

## Install Requirements

In [ ]:
!nvidia-smi

In [ ]:
pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install natten==0.17.5+torch250cu124 -f https://whl.natten.org

Clone the repository and move into it so `MedViT.py` and `datasets.py` are importable:

In [ ]:
!git clone https://github.com/Omid-Nejati/MedViTV2.git
%cd /content/MedViTV2

In [ ]:
!pip install -r requirements.txt

## Imports

The pipeline's own imports, plus `build_dataset` (from `datasets.py`) and the four `MedViT_*`
builders (from `MedViT.py`) — both from the cloned repo — and the reusable helpers from
`_handlers/evaluation.py`. The `sys.path` walk locates the survey's `_handlers` package; if you run
this in Colab after `%cd`-ing into MedViTV2, make sure the survey `src` is reachable (or add it to
`sys.path` manually).

In [ ]:
import pathlib
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from medmnist import INFO
from datasets import build_dataset
from MedViT import MedViT_tiny, MedViT_small, MedViT_base, MedViT_large

# put the survey `_handlers` package on the path, then import the reusable pipeline
for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_root / "_handlers").is_dir():
        sys.path.insert(0, str(_root))
        break

from _handlers.evaluation import (
    build_model,
    train_mnist,
    train_other,
    evaluate_all_metrics,
)

## Configuration

`main.py` reads its settings from `argparse` on the command line. Here we replace that block with a
plain config object, so the exact same `args` flows through the pipeline.

**Model** — `[MedViT_tiny, MedViT_small, MedViT_base, MedViT_large]`, or any [timm](https://timm.fast.ai/)
model name (e.g. `resnet34`, `convnext_tiny`, `vit_base_patch16_224`), which falls through to the
`timm.create_model` branch.

**Dataset** — MedMNIST: `tissuemnist, pathmnist, chestmnist, dermamnist, octmnist, pneumoniamnist,
retinamnist, breastmnist, bloodmnist, organamnist, organcmnist, organsmnist`; or image-folder
datasets: `Kvasir, CPN, Fetal, PAD, ISIC2018`. The first download can take a while.

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    model_name='MedViT_small',                       # a MedViT_* key or any timm model
    dataset='breastmnist',                           # a *mnist flag or an image-folder dataset
    batch_size=24,
    lr=1e-4,
    epochs=100,
    pretrained=False,                                # load the pretrained MedViT checkpoint
    checkpoint_path='./checkpoint/MedViT_small.pth', # used only when pretrained=True
)

## Device

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

## Dataset

For MedMNIST datasets the `task` field (multi-label vs. multi-class) selects the loss. `build_dataset`
then downloads the data, resizes to 224×224, applies the transforms, and returns the datasets plus
the number of classes.

In [ ]:
# medmnist datasets carry a `task` that selects the loss; image-folder datasets are plain multi-class
if args.dataset.endswith('mnist'):
    info = INFO[args.dataset]
    task = info['task']
    if task == "multi-label, binary-class":
        loss_function = nn.BCEWithLogitsLoss()
    else:
        loss_function = nn.CrossEntropyLoss()
else:
    task = None
    loss_function = nn.CrossEntropyLoss()

train_dataset, test_dataset, nb_classes = build_dataset(args=args)

print(train_dataset)
print("===================")
print(test_dataset)

## Model

`model_classes` maps the `MedViT_*` names to the builders imported from the cloned repo — the one
piece that must stay in the notebook. `build_model` (in the handler) uses it: a known name builds that
MedViT (optionally loading a pretrained checkpoint and dropping the head when the class count
differs); any other name falls through to `timm.create_model`, so the same call trains a CNN
(`resnet34`, `convnext_tiny`, …) or a transformer (`vit_base_patch16_224`,
`swin_tiny_patch4_window7_224`, …).

In [ ]:
model_classes = {
    'MedViT_tiny': MedViT_tiny,
    'MedViT_small': MedViT_small,
    'MedViT_base': MedViT_base,
    'MedViT_large': MedViT_large,
}

net = build_model(args.model_name, nb_classes, model_classes,
                  pretrained=args.pretrained, checkpoint_path=args.checkpoint_path)

## Optimizer, Scheduler & Data Loaders

AdamW with weight decay and a cosine-annealing schedule stepped **every iteration** — so `T_max` is
the total number of optimizer steps (`epochs * train_num // batch_size`).

In [ ]:
train_num = len(train_dataset)
eta = args.epochs * train_num // args.batch_size   # total scheduler steps

optimizer = optim.AdamW(net.parameters(), lr=args.lr, betas=[0.9, 0.999], weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=eta, eta_min=5e-6)

train_loader = data.DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*args.batch_size, shuffle=False)

## Train

Pick the routine that matches the dataset — `train_mnist` (medmnist Evaluator) or `train_other`
(scikit-learn metrics), both imported from the handler. Each evaluates every epoch and writes the
best model to `save_path`.

In [ ]:
save_path = f'./{args.model_name}_{args.dataset}.pth'

if args.dataset.endswith('mnist'):
    train_mnist(args.epochs, net, train_loader, test_loader,
                optimizer, scheduler, loss_function, device, save_path, args.dataset, task)
else:
    train_other(args.epochs, net, train_loader, test_loader,
                optimizer, scheduler, loss_function, device, save_path)

## Evaluate all metrics

`evaluate_all_metrics` (from the handler) runs the model once over one split and prints **every**
metric the two routines can produce: the medmnist Evaluator AUC/ACC (for `*mnist` datasets) plus
accuracy, weighted precision / recall (sensitivity) / F1, per-class + average specificity,
one-vs-rest AUC, the confusion matrix and a per-class report. Set `split` to `'train'` or `'test'`;
uncomment the `load_state_dict` line to score the best checkpoint instead of the in-memory model.

In [ ]:
split = 'test'   # 'train' or 'test'

# net.load_state_dict(torch.load(save_path)['model'])   # uncomment to evaluate the BEST checkpoint

eval_dataset = train_dataset if split == 'train' else test_dataset
metrics = evaluate_all_metrics(net, eval_dataset, args.dataset, nb_classes, device,
                               split=split, batch_size=2 * args.batch_size)